### ЗАДАЧА: Панель контроля рекламных бюджетов по паттерну `MVC`

Маркетинговая команда следит за рекламными кампаниями и проверяет кейсы с перерасходом.
Для каждого кейса нужно хранить:
- `case_id`
- `campaign`
- `channel`
- `planned_budget`
- `spent_budget`
- `overspend_amount` — вычисляемое поле `spent_budget - planned_budget`
- `status`
- `manager`
- `resolution`

Статусы кейса:
- `new`
- `reviewing`
- `approved`
- `frozen`

Бизнес-правила:
- нельзя создать кейс с уже существующим `case_id`;
- при создании нужно корректно считать `overspend_amount`;
- начать review можно только если назначен `manager`;
- одобрить кейс можно только из `reviewing`;
- заморозить кампанию можно только из `reviewing`;
- финальные кейсы (`approved`, `frozen`) нельзя менять дальше;
- при `approved` и `frozen` нужно записывать `resolution`.

In [1]:
from dataclasses import dataclass


initial_cases = [
    ("PB-100", "spring_sale", "google_ads", 1000.0, 1450.0),
    ("PB-101", "brand_push", "yandex_ads", 800.0, 760.0),
]

actions = [
    ("show",),
    ("review", "PB-100"),
    ("assign", "PB-100", "Olga"),
    ("review", "PB-100"),
    ("approve", "PB-100", "budget_extended"),
    ("create", "PB-102", "new_arrivals", "tiktok_ads", 500.0, 950.0),
    ("assign", "PB-102", "Max"),
    ("freeze", "PB-102", "overspend_blocked"),
    ("show",),
]


@dataclass
class PromoBudgetCase:
    case_id: str
    campaign: str
    channel: str
    planned_budget: float
    spent_budget: float
    overspend_amount: float
    status: str = "new"
    manager: str | None = None
    resolution: str | None = None


class PromoBudgetModel:
    def __init__(self):
        self.cases = {}

    def _calculate_overspend(self, planned_budget: float, spent_budget: float) -> float:
        return spent_budget - planned_budget
    
    def _check_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError('Case not found')
        return self.cases[case_id]
    
    
    def add_case(self, case_id: str, campaign: str, channel: str, planned_budget: float, spent_budget: float) -> None:
        overspend_amount = self._calculate_overspend(planned_budget, spent_budget)
        if case_id in self.cases:
            raise ValueError('Case already exists')
        self.cases[case_id] = PromoBudgetCase(case_id, campaign, channel, planned_budget, spent_budget, overspend_amount)

    def assign_manager(self, case_id: str, manager: str) -> None:
        case = self._check_case(case_id)
        if case.status in {'approved', 'frozen'}:
            raise ValueError('Cannot change cases with final statuses')
        case.manager = manager

    def start_review(self, case_id: str) -> None:
        case = self._check_case(case_id)
        if case.status in {'approved', 'frozen'}:
            raise ValueError('Cannot change cases with final statuses')
        if case.manager is None:
            raise ValueError('Manager is required')
        case.status = "reviewing"

    def approve_case(self, case_id: str, resolution: str) -> None:
        case = self._check_case(case_id)
        if case.status != 'reviewing':
            raise ValueError('"Reviewing" status is required')
        case.status = "approved"
        case.resolution = resolution

    def freeze_case(self, case_id: str, resolution: str) -> None:
        case = self._check_case(case_id)
        if case.status != 'reviewing':
            raise ValueError('"Reviewing" status is required')
        case.status = "frozen"
        case.resolution = resolution

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.campaign} | {case.channel} | {case.planned_budget:.2f} | "
                f"{case.spent_budget:.2f} | {case.overspend_amount:.2f} | {case.status} | {case.manager} | {case.resolution}"
            )
        return rows


class PromoBudgetView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Promo budget cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class PromoBudgetController:
    def __init__(self, model: PromoBudgetModel, view: PromoBudgetView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, campaign: str, channel: str, planned_budget: float, spent_budget: float) -> None:
        try:
            self.model.add_case(case_id, campaign, channel, planned_budget, spent_budget)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_manager(self, case_id: str, manager: str) -> None:
        try:
            self.model.assign_manager(case_id, manager)
            self.view.render_success(f"Manager assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_review(self, case_id: str) -> None:
        try:
            self.model.start_review(case_id)
            self.view.render_success(f"Case {case_id} moved to review")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_case(self, case_id: str, resolution: str) -> None:
        try:
            self.model.approve_case(case_id, resolution)
            self.view.render_success(f"Case {case_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))

    def freeze_case(self, case_id: str, resolution: str) -> None:
        try:
            self.model.freeze_case(case_id, resolution)
            self.view.render_success(f"Case {case_id} frozen")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())


model = PromoBudgetModel()
view = PromoBudgetView()
controller = PromoBudgetController(model, view)

for case_id, campaign, channel, planned_budget, spent_budget in initial_cases:
    model.add_case(case_id, campaign, channel, planned_budget, spent_budget)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, campaign, channel, planned_budget, spent_budget = action
        controller.create_case(case_id, campaign, channel, planned_budget, spent_budget)
    elif action[0] == "assign":
        _, case_id, manager = action
        controller.assign_manager(case_id, manager)
    elif action[0] == "review":
        _, case_id = action
        controller.start_review(case_id)
    elif action[0] == "approve":
        _, case_id, resolution = action
        controller.approve_case(case_id, resolution)
    elif action[0] == "freeze":
        _, case_id, resolution = action
        controller.freeze_case(case_id, resolution)

print()
print("Финальное состояние:")
controller.show_cases()


Promo budget cases:
PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | new | None | None
PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
ERROR: Manager is required
SUCCESS: Manager assigned to PB-100
SUCCESS: Case PB-100 moved to review
SUCCESS: Case PB-100 approved
SUCCESS: Case PB-102 created
SUCCESS: Manager assigned to PB-102
ERROR: "Reviewing" status is required
Promo budget cases:
PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None

Финальное состояние:
Promo budget cases:
PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None


In [ ]:
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | new | None | None
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# ERROR: Manager is required
# SUCCESS: Manager assigned to PB-100
# SUCCESS: Case PB-100 moved to review
# SUCCESS: Case PB-100 approved
# SUCCESS: Case PB-102 created
# SUCCESS: Manager assigned to PB-102
# ERROR: Case is not in review
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None
#
# Финальное состояние:
# Promo budget cases:
# PB-100 | spring_sale | google_ads | 1000.00 | 1450.00 | 450.00 | approved | Olga | budget_extended
# PB-101 | brand_push | yandex_ads | 800.00 | 760.00 | -40.00 | new | None | None
# PB-102 | new_arrivals | tiktok_ads | 500.00 | 950.00 | 450.00 | new | Max | None